In [ ]:
# Import Relevant Modules

import sys
import numpy as np
import pandas as pd
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sb
import cvxpy as cp
import math
#!{sys.executable} -m pip install yfinance
import yfinance as yf
#!{sys.executable} -m pip install pandas_datareader
from pandas_datareader import data as web

print("USING:", sys.executable)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)

# IT WAS ALSO POSSIBLE TO USE sklearn:
#from sklearn.linear_model import ElasticNetCV
#import pandas as pd
#import numpy as np
#import datetime as dt
# from scipy import minimize
# from sklearn.linear_model import ElasticNetCV


# Import Relevant Modules
# If cvxpy is required, install it in a separate notebook cell:

#import cvxpy as cp

USING: c:\Users\User\quant39\Scripts\python.exe
numpy: 2.0.2
pandas: 2.3.3


In [ ]:
# Apple, Microsoft, JPMorgan Chase, Exxon Mobil, Johnson & Johnson, Walmart, Coca-Cola, 
# Procter & Gamble, Caterpillar, IBM stock tickers.
tickers = ["AAPL","MSFT","JPM","XOM","JNJ","WMT","KO","PG","CAT","IBM"]

# Download adjusted daily stock prices.
downloaded_data = yf.download(
    tickers=tickers,
    start="1992-12-01",
    end="2026-01-01",
    auto_adjust=True,
    progress=False
)

# Use daily closing prices and reindex to ensure all tickers are present.
daily_prices = downloaded_data["Close"]
daily_prices = daily_prices.reindex(columns=tickers)


# Convert daily prices to monthly prices and returns.
monthly_prices = daily_prices.resample("ME").last()
monthly_returns = monthly_prices.pct_change(fill_method=None)
monthly_returns.index = monthly_returns.index.to_period("M")
monthly_returns = monthly_returns.loc["1993-01":"2025-12"]

# Test the monthly returns data by printing
print(monthly_returns.head())


In [ ]:
# Read in data from K French Library
ff_data = pd.read_csv('F-F_Research_Data_5_Factors_2x3.csv')
lt_rev_data = pd.read_csv('F-F_LT_Reversal_Factor.csv')
st_rev_data = pd.read_csv('F-F_ST_Reversal_Factor.csv')
mom_data = pd.read_csv('F-F_Momentum_Factor.csv')

# Rename date column
ff_data.rename(columns={'Unnamed: 0': 'Date'}, inplace=True)
lt_rev_data.rename(columns={'Unnamed: 0': 'Date1'}, inplace=True)
st_rev_data.rename(columns={'Unnamed: 0': 'Date2'}, inplace=True)
mom_data.rename(columns={'Unnamed: 0': 'Date3'}, inplace=True)

# Combine factor returns for first dataset
combined_all = pd.concat([ff_data, lt_rev_data, st_rev_data, mom_data], axis=1)
combined_all.to_csv('combined_factors.csv', index=False)

# Remove redundant date columns and reorder
combined_all.drop(columns=['Date1', 'Date2', 'Date3'], inplace=True)

# Create factor return matrix by dropping date and creating (Mkt-Rf) as the first column
# factor_returns_only_d1 = combined_all.drop(columns=['Date', 'RF'])

# Check the data
combined_all.head()
# factor_returns_only_d1.head()

In [ ]:
# Filter to get dataset 1 (1993-2003)
combined_d1 = combined_all.loc[(pd.to_numeric(combined_all['Date'], errors='coerce') >= 199301) 
                 & (pd.to_numeric(combined_all['Date'], errors='coerce') <= 200312)]
factor_returns_d1 = combined_d1.drop(columns=['Date', 'RF'])
risk_free_rate_d1 = combined_d1['RF'].astype(float) / 100

# Filter to get dataset 2 (2004-2014)
combined_d2 = combined_all.loc[(pd.to_numeric(combined_all['Date'], errors='coerce') >= 200401) 
                 & (pd.to_numeric(combined_all['Date'], errors='coerce') <= 201412)]
factor_returns_d2 = combined_d2.drop(columns=['Date', 'RF'])
risk_free_rate_d2 = combined_d2['RF'].astype(float) / 100

# Filter to get dataset 3 (2015-2025)
combined_d3 = combined_all.loc[(pd.to_numeric(combined_all['Date'], errors='coerce') >= 201501) 
                 & (pd.to_numeric(combined_all['Date'], errors='coerce') <= 202512)]
factor_returns_d3 = combined_d3.drop(columns=['Date', 'RF'])
risk_free_rate_d3 = combined_d3['RF'].astype(float) / 100

# factor_returns_d1.head()
mask = factor_returns_d1.eq("Mkt-RF")
print(mask.any())

rows, cols = np.where(mask)

for r, c in zip(rows, cols):
    print("Row:", r)
    print("Column:", factor_returns_d1.columns[c])

In [ ]:

# Check filtered data for datasets
# combined_d1.head()
factor_returns_d1 = factor_returns_d1.astype(float)
factor_returns_d1 = factor_returns_d1 / 100
# combined_d2.head()
factor_returns_d2 = factor_returns_d2.astype(float)
factor_returns_d2 = factor_returns_d2 / 100
# combined_d3.head()
factor_returns_d3 = factor_returns_d3.astype(float)
factor_returns_d3 = factor_returns_d3 / 100
factor_returns_d1.head()


# Convert monthly returns to excess returns by subtracting the 
# risk-free rate from the monthly returns for each dataset
excess_returns_d1 = monthly_returns.loc["1993-01":"2003-12"].sub(risk_free_rate_d1.values / 100, axis=0)
excess_returns_d2 = monthly_returns.loc["2004-01":"2014-12"].sub(risk_free_rate_d2.values / 100, axis=0)
excess_returns_d3 = monthly_returns.loc["2015-01":"2025-12"].sub(risk_free_rate_d3.values / 100, axis=0)

monthly_returns_d1 = monthly_returns.loc["1993-01":"2003-12"]
monthly_returns_d2 = monthly_returns.loc["2004-01":"2014-12"]
monthly_returns_d3 = monthly_returns.loc["2015-01":"2025-12"]

# Check the excess returns for dataset 1
excess_returns_d1.head()


In [ ]:
# For each base model, define a function to obtain the
# factor loadings

# OLS factor loadings
def OLS_get_factor_loadings(factor_returns, excess_returns):
    """
    Parameters:
    ___________
    factor_returns: np.ndarray
        Matrix of historical factor returns of shape (T,K)
        where T is the number of periods in the sample and
        K is the number of factors considered
    ___________
    excess_returns : np.ndarray
        Matrix of historical excess asset returns of shape
        (T,N) where T is as above and N is the number of 
        assets
    """
    # Convert factor returns and excess returns to numpy arrays
    F = np.asarray(factor_returns); y = np.asarray(excess_returns)
    
    # Establish time periods in sample, number of assets, number of factors
    T = F.shape[0]
    N = y.shape[1]
    K = F.shape[1]

    # Check each data array has T time periods in the sample
    assert(F.shape[0] == y.shape[0])

    # Add column of ones corresponding to intercept
    X = np.column_stack((np.ones(T), F))

    # Compute OLS Solution : (X^T @ X)^{-1} @ X^T y
    factor_loadings = np.linalg.inv(X.T @ X) @ X.T @ y

    # Split factor loadings into alpha and beta
    alpha = factor_loadings[0,:]
    beta = factor_loadings[1:,:]

    return alpha, beta

In [166]:

# 3-Factor Fama-French factor loadings
def FF3_get_factor_loadings(factor_returns, excess_returns):
    """
    Parameters:
    ___________
    factor_returns: np.ndarray
        Matrix of historical factor returns of shape (T,K)
        where T is the number of periods in the sample and
        K is the number of factors considered
        It is assumed the first three columns are named
        'Mkt-Rf', 'HML', 'SMB' exactly
    ___________
    excess_returns : np.ndarray
        Matrix of historical excess asset returns of shape
        (T,N) where T is as above and N is the number of 
        assets
    """
    # Resize the factor_returns matrix to only contain 3-Factor
    # Fama-French factors: Mkt-Rf, HML, SMB
    three_factor_returns = factor_returns[:,[0,1,2]]

    # Use existing OLS method to obtain the factor loadings for
    # 3-Factor Fama-French Model
    return OLS_get_factor_loadings(three_factor_returns, excess_returns)



In [ ]:
def EN_get_factor_loadings(factor_returns, excess_returns, opt_lambda_1, opt_lambda_2):
    """
    Elastic Net factor loadings solver.

    NOTE: `factor_returns` MUST be pre-standardized (zero mean, unit std) before calling.
    Parameters:
      factor_returns: np.ndarray, shape (T, K) - standardized
      excess_returns: np.ndarray, shape (T, N)
      opt_lambda_1: float - L1 penalty weight
      opt_lambda_2: float - L2 penalty weight (squared)
    Returns:
      alpha: (N,) intercepts
      beta: (K, N) factor loadings
      """
      
    # Convert factor returns and excess asset returns to numpy arrays
    F = np.asarray(factor_returns, dtype=float)
    y = np.asarray(excess_returns, dtype=float)

    # Set dimensions
    T = F.shape[0]
    N = y.shape[1]
    K = F.shape[1]

    assert F.shape[0] == y.shape[0], "factor_returns and excess_returns must have same T."

    # Design matrix with intercept
    X = np.column_stack((np.ones(T), F))

    # cvxpy variable: (K+1, N)
    beta = cp.Variable((K + 1, N))

    # Objective: sum of squared errors + L1 on loadings (exclude intercept) + L2 (squared Frobenius) on loadings
    residual_term = cp.sum_squares(X @ beta - y)
    l1_term = opt_lambda_1 * cp.norm1(beta[1:, :])
    l2_term = opt_lambda_2 * cp.sum_squares(beta[1:, :])

    # Define the optimization problem
    objective = cp.Minimize(residual_term + l1_term + l2_term)
    problem = cp.Problem(objective)

    # Solve the problem using SCS solver for stability
    problem.solve(solver=cp.SCS, verbose=False)  # explicit solver for stability

    alpha = beta.value[0, :].reshape(-1)
    loadings = beta.value[1:, :]

    return alpha, loadings

    

In [ ]:
def get_lambdas(excess_returns, factor_returns):
    """
    Parameters:
    factor_returns: np.ndarray, shape (T, K)
        Matrix of historical factor returns of shape (T,K)
    excess_returns: np.ndarray, shape (T, N)
        Matrix of historical excess asset returns of shape

    Returns:
    best_lambda_1: float - best L1 penalty weight
    best_lambda_2: float - best L2 penalty weight
    results: list of dictionaries containing lambda_1, lambda_2, average_rmse, and fold_rmses for each combination
    """

    # Set a grid of lambda values raning from 2^-6 to 2^3 for both L1 and L2 penalties
    lambda1_values = np.logspace(-6, 3, base=2, num=10)
    lambda2_values = np.logspace(-6, 3, base=2, num=10)

    # Convert factor_returns and excess_returns to numpy arrays of type float
    F = np.asarray(factor_returns,dtype=float)
    Y = np.asarray(excess_returns,dtype=float)

    # Check for NaN or inf values in the input arrays
    if not np.all(np.isfinite(F)):
        raise ValueError("factor_returns contains NaN or inf.")
    if not np.all(np.isfinite(Y)):
        raise ValueError("excess_returns contains NaN or inf.")

    # Set up 4 folds for cross-validation: (train_start, train_end, validation_start, validation_end)
    folds = [(0, 36, 36, 42),(6, 42, 42, 48),(12, 48, 48, 54),(18, 54, 54, 60)]

    # Initialize variables to track the best lambda values and the corresponding average RMSE
    best_lambda_1 = None
    best_lambda_2 = None
    best_average_rmse = np.inf

    results = []

    # Run the 4-fold validation for each combination of lambda_1 and lambda_2
    for lambda_1 in lambda1_values:
        for lambda_2 in lambda2_values:
            fold_rmses = []
            for (train_start,train_end,
                validation_start,validation_end) in folds:

                F_train = F[train_start:train_end]
                Y_train = Y[train_start:train_end]

                F_validation = F[validation_start:validation_end]
                Y_validation = Y[validation_start:validation_end]

                # Standardize factor returns
                F_mean = np.mean(F_train,axis=0)
                F_std = np.std(F_train,axis=0,ddof=1)
                F_std = np.where(F_std == 0,1.0,F_std)

                F_train_scaled = (F_train - F_mean) / F_std
                F_validation_scaled = (F_validation - F_mean) / F_std

                # Fit Elastic Net for each lambda_1 and lambda_2
                intercepts, loadings = (EN_get_factor_loadings(F_train_scaled,
                        Y_train,lambda_1,lambda_2))

                # Compute the predicted returns for the validation
                predicted_returns = (F_validation_scaled @ loadings + intercepts)

                # Compute the RMSE for the current fold
                fold_rmse = np.sqrt(
                    np.mean((Y_validation - predicted_returns) ** 2))

                fold_rmses.append(fold_rmse)

            # Compute mean RMSE across all folds for the current lambda_1 and lambda_2
            average_rmse = np.mean(fold_rmses)

            results.append({"lambda_1":lambda_1,
                "lambda_2":lambda_2,"average_rmse":average_rmse,
                "fold_rmses":fold_rmses})

            if (average_rmse < best_average_rmse):
                best_average_rmse = (average_rmse)
                best_lambda_1 = (lambda_1)
                best_lambda_2 = (lambda_2)

    return (best_lambda_1,best_lambda_2,results)

In [ ]:

def get_mean_cov(alpha, B, factor_returns, excess_returns, holding_period):
    """
    Parameters:
    Estimate the expected asset returns and asset covariance matrix.
    alpha : np.ndarray
        Intercept vector of shape (N,).
    B : np.ndarray
        Factor-loading matrix of shape (K, N).
    factor_returns : np.ndarray
        Factor-return matrix of shape (T, K).
    excess_returns : np.ndarray
        Asset excess-return matrix of shape (T, N).
    holding_period : int
        The holding period for the factor model.

    Returns:
    mean_vector : np.ndarray
        Expected excess-return vector of shape (N,).
    Q : np.ndarray
        Asset covariance matrix of shape (N, N).
    """

    # Convert inputs to numpy arrays and ensure correct shapes
    alpha = np.asarray(alpha, dtype=float).reshape(-1)
    B = np.asarray(B, dtype=float)
    factor_returns = np.asarray(factor_returns, dtype=float)
    excess_returns = np.asarray(excess_returns, dtype=float)

    T, K = factor_returns.shape
    N = excess_returns.shape[1]

    # Expected asset returns:
    # E[r] = alpha + B' E[f]
    F_bar_36 = np.mean(factor_returns[(T- 6*holding_period):, :], axis=0)

    F_bar = np.mean(factor_returns[(T- 6*holding_period):, :], axis=0)

    mean_vector = alpha + B.T @ F_bar_36

    # Factor covariance matrix, shape (K, K)
    centered_factors = factor_returns - F_bar

    F = (centered_factors.T@ centered_factors) / (T - 1)

    # Predicted historical returns, shape (T, N)
    predicted_returns = alpha + factor_returns @ B

    # Regression residuals, shape (T, N)
    residuals = excess_returns - predicted_returns

    # Diagonal residual covariance
    residual_variances = np.sum(residuals ** 2,axis=0) / (T - K - 1)
    
    E = np.diag(residual_variances)

    # Asset covariance matrix, shape (N, N)
    Q = B.T @ F @ B + E

    return mean_vector, Q



In [ ]:
def get_return_prediction_error(
    alpha_1, alpha_2, alpha_3,
    B_1, B_2, B_3,
    factor_returns,
    excess_returns, lambda_w,
    holding_period):
    """
    Calculate the mean squared prediction error for a given set of factor loadings.
    alpha : np.ndarray
        Intercept terms for each asset, shape (N,).
    B_1, B_2, B_3 : np.ndarray
        Factor loadings for each asset and factor, shape (K, N)
        over the last 3 rebalancing periods.
    factor_returns : np.ndarray
        Factor returns over the period, shape (T, K).
    excess_returns : np.ndarray
        Excess returns over the period, shape (T, N).
        lambda_w: float
        Exponential decay weight for the MSEs of the last 3 periods.
    """
    
    alpha_1 = np.asarray(alpha_1, dtype=float).reshape(-1)
    alpha_2 = np.asarray(alpha_2, dtype=float).reshape(-1)
    alpha_3 = np.asarray(alpha_3, dtype=float).reshape(-1)
    B_1 = np.asarray(B_1, dtype=float)
    B_2 = np.asarray(B_2, dtype=float)
    B_3 = np.asarray(B_3, dtype=float)
    factor_returns = np.asarray(factor_returns, dtype=float)
    excess_returns = np.asarray(excess_returns, dtype=float)

    # Obtain the excess returns and factor returns for the last 3 rebalancing periods
    excess_returns_1 = excess_returns[-holding_period:, :]
    excess_returns_2 = excess_returns[-2*holding_period:-holding_period, :]
    excess_returns_3 = excess_returns[-3*holding_period:-2*holding_period, :]

    factor_returns_1 = factor_returns[-holding_period:, :]
    factor_returns_2 = factor_returns[-2*holding_period:-holding_period, :]
    factor_returns_3 = factor_returns[-3*holding_period:-2*holding_period, :]

    # Predicted returns:
    # alpha has shape (N,)
    # factor_returns @ B has shape (K, N)
    predicted_mean_1 = alpha_1 + np.mean(factor_returns_1, axis=0) @ B_1
    predicted_mean_2 = alpha_2 + np.mean(factor_returns_2, axis=0) @ B_2
    predicted_mean_3 = alpha_3 + np.mean(factor_returns_3, axis=0) @ B_3

    errors_1 = np.mean(excess_returns_1, axis=0) - predicted_mean_1
    errors_2 = np.mean(excess_returns_2, axis=0) - predicted_mean_2
    errors_3 = np.mean(excess_returns_3, axis=0) - predicted_mean_3

    # Compute MSEs using past 3 predicted means vs. past 3 sample means
    mean_squared_error_1 = np.mean(errors_1 ** 2)
    mean_squared_error_2 = np.mean(errors_2 ** 2)
    mean_squared_error_3 = np.mean(errors_3 ** 2)

    # Apply exponential decay methodology to weight MSEs prior to
    # converting to RMSE form
    
    weights = np.zeros(3)
    for i in range(0,3):
        weights[i] = (1-lambda_w)*(lambda_w**i)

    # Normalize the weights
    weights = weights / np.sum(weights)

    # Calculate the weighted MSE
    weighted_error = (
        mean_squared_error_1*weights[0] + mean_squared_error_2*weights[1] + 
        mean_squared_error_3*weights[2]) 

    # Calculate the RMSE of the weighted error
    rmse_weighted_error = np.sqrt(weighted_error)

    return rmse_weighted_error


'\n\n\ndef get_cov_prediction_error(predicted_cov, last_excess_returns):\n    \n    Calculate the RMSE between a model\'s predicted covariance matrix\n    and the realized sample covariance matrix over the previous\n    holding period.\n\n    Parameters\n    ----------\n    predicted_cov : np.ndarray\n        Predicted covariance matrix from the previous rebalance,\n        shape (N, N).\n\n    last_excess_returns : np.ndarray\n        Excess returns over the previous holding period,\n        shape (H, N).\n\n    Returns\n    -------\n    float\n        RMSE between the predicted and realized covariance matrices.\n    \n\n    predicted_cov = np.asarray(predicted_cov, dtype=float)\n    last_excess_returns = np.asarray(last_excess_returns, dtype=float)\n\n    # Sample covariance matrix over the previous holding period\n    realized_cov = np.cov(\n        last_excess_returns,\n        rowvar=False,\n        ddof=1\n    )\n\n    if predicted_cov.shape != realized_cov.shape:\n        raise 

In [ ]:
def MVO(mean_returns,cov,prev_weights,Transaction_costs):

    # Set returns and covariance matrix as numpy arrays
    mean_returns = np.asarray(mean_returns,dtype=float).reshape(-1)
    cov = np.asarray(cov,dtype=float)

    prev_weights = np.asarray(prev_weights,dtype=float).reshape(-1)

    # Set the number of assets
    n = len(mean_returns)

    # Make sure covariance matrix is symmetric
    cov = 0.5 * (cov + cov.T)

    
    # Max feasible net expected return
    w_max = cp.Variable(n)

    max_return_problem = cp.Problem(
        cp.Maximize(
            mean_returns @ w_max - Transaction_costs
            * cp.sum(cp.abs(w_max - prev_weights))), [cp.sum(w_max) == 1, w_max >= 0])

    max_return_problem.solve(verbose=False)

    max_feasible_return = (
        mean_returns @ w_max.value
        - Transaction_costs
        * np.sum(np.abs(w_max.value- prev_weights)))

    # Original desired target
    desired_target = np.mean(
        mean_returns)
    # Make target feasible and adjust if needed
    epsilon = 1e-8

    target_return = min(desired_target,max_feasible_return - epsilon)

    if desired_target <= max_feasible_return:
        target_return = desired_target
        target_adjusted = False
    else:
        target_return = max_feasible_return - 1e-4
        target_adjusted = True

    if target_adjusted:
        print("Target adjusted")

    # Step 2: Minimum variance MVO- classic setup with transaction costs and target return constraint
    x = cp.Variable(n)

    problem = cp.Problem(
        cp.Minimize(
            0.5 * cp.quad_form(
                x,cov)),
        [mean_returns @ x
            - Transaction_costs
            * cp.sum(cp.abs(x - prev_weights))
            >= target_return,
            cp.sum(x) == 1, x >= 0])

    problem.solve(verbose=False)

    return np.asarray(x.value).reshape(-1)

In [172]:
# Define quick helper functions

def drift_weights(weights, asset_returns):
    ending_values = weights * (1.0 + asset_returns)
    return ending_values / ending_values.sum()

def calculate_turnover(new_weights, current_weights):
    return 0.5 * np.sum(np.abs(new_weights - current_weights))

In [ ]:
def rebalancing(train_factor_returns, train_excess_returns, train_abs_returns, prev_weights, Transaction_costs, 
                Temperature_M, beta_OLS_prev, beta_FF3_prev, beta_EN_prev,
                alpha_OLS_prev, alpha_FF3_prev, alpha_EN_prev, 
                beta_OLS_prev_2, beta_FF3_prev_2, beta_EN_prev_2,
                alpha_OLS_prev_2, alpha_FF3_prev_2, alpha_EN_prev_2,
                beta_OLS_prev_3, beta_FF3_prev_3, beta_EN_prev_3,
                alpha_OLS_prev_3, alpha_FF3_prev_3, alpha_EN_prev_3, is_first,
                equal_weight, OLS_test, FF3_test, EN_test, holding_period, lambda_w,
                opt_lambda_1=None, opt_lambda_2=None):
    """
    Compute ensemble allocation and return model parameters and model weights.
    Returns allocation and many diagnostic items including both mean-weights and cov-weights.
    """
    # Default covariance weights (can be adapted later)
    OLS_cov_weight = 1/3
    FF3_cov_weight = 1/3
    EN_cov_weight = 1/3

    # Precompute standardized factors for Elastic Net (EN expects standardized factors)
    EN_factor_mean = np.mean(train_factor_returns, axis=0)
    EN_factor_std = np.std(train_factor_returns, axis=0, ddof=1)
    EN_factor_std = np.where(EN_factor_std == 0, 1.0, EN_factor_std)
    train_factor_returns_EN = (train_factor_returns - EN_factor_mean) / EN_factor_std

    # Compute factor loadings for all base models
    alpha_OLS, beta_OLS = OLS_get_factor_loadings(train_factor_returns, train_excess_returns)
    mean_OLS, cov_OLS = get_mean_cov(alpha_OLS, beta_OLS, train_factor_returns, train_excess_returns, holding_period)

    alpha_FF3, beta_FF3 = FF3_get_factor_loadings(train_factor_returns, train_excess_returns)
    three_factor_returns = train_factor_returns[:, [0, 1, 2]]
    mean_FF3, cov_FF3 = get_mean_cov(alpha_FF3, beta_FF3, three_factor_returns, train_excess_returns, holding_period)

    # If lambdas were not provided, compute them using raw (unstandardized) train_factor_returns
    if opt_lambda_1 is None or opt_lambda_2 is None:
        opt_lambda_1, opt_lambda_2, _ = get_lambdas(train_excess_returns, train_factor_returns)

    # EN expects standardized factors
    alpha_EN, beta_EN = EN_get_factor_loadings(train_factor_returns_EN, train_excess_returns, 
                                               opt_lambda_1, opt_lambda_2)
    mean_EN, cov_EN = get_mean_cov(alpha_EN, beta_EN, train_factor_returns_EN, train_excess_returns, holding_period)

    # Determine mean-weights (softmax over recent performance) if we have history and not testing fixed models
    have_three_previous_models = (beta_OLS_prev is not None and beta_OLS_prev_2 is not None
                                and beta_OLS_prev_3 is not None)

    if have_three_previous_models and not equal_weight and not OLS_test and not FF3_test and not EN_test:
        OLS_prev_Merror = get_return_prediction_error(
            alpha_OLS_prev, alpha_OLS_prev_2, alpha_OLS_prev_3, beta_OLS_prev, beta_OLS_prev_2, beta_OLS_prev_3,
            train_factor_returns, train_excess_returns, lambda_w, holding_period)
        FF3_prev_Merror = get_return_prediction_error(
            alpha_FF3_prev, alpha_FF3_prev_2, alpha_FF3_prev_3, beta_FF3_prev, beta_FF3_prev_2, beta_FF3_prev_3,
            train_factor_returns[:, [0, 1, 2]], train_excess_returns, lambda_w, holding_period)
        EN_prev_Merror = get_return_prediction_error(
            alpha_EN_prev, alpha_EN_prev_2, alpha_EN_prev_3, beta_EN_prev, beta_EN_prev_2, beta_EN_prev_3,
            train_factor_returns_EN, train_excess_returns, lambda_w, holding_period)

        Merror_mean = np.mean([OLS_prev_Merror, FF3_prev_Merror, EN_prev_Merror])
        Merror_std = np.std([OLS_prev_Merror, FF3_prev_Merror, EN_prev_Merror])
        epsilon = 1e-8

        std_OLS_prev_Merror = (OLS_prev_Merror - Merror_mean) / (Merror_std + epsilon)
        std_FF3_prev_Merror = (FF3_prev_Merror - Merror_mean) / (Merror_std + epsilon)
        std_EN_prev_Merror = (EN_prev_Merror - Merror_mean) / (Merror_std + epsilon)

        recip_exp_mean = 1 / (math.exp(-1 * std_OLS_prev_Merror * Temperature_M)
                              + math.exp(-1 * std_FF3_prev_Merror * Temperature_M)
                              + math.exp(-1 * std_EN_prev_Merror * Temperature_M))
        
        OLS_mean_weight = math.exp(-1 * Temperature_M * std_OLS_prev_Merror) * recip_exp_mean
        FF3_mean_weight = math.exp(-1 * Temperature_M * std_FF3_prev_Merror) * recip_exp_mean
        EN_mean_weight = math.exp(-1 * Temperature_M * std_EN_prev_Merror) * recip_exp_mean
        is_first = False

        # Test the mean return vectors for each base model
        """
        print("OLS mean:"); print(mean_OLS)
        print("FF3 mean:"); print(mean_FF3)
        print("EN mean:"); print(mean_EN) 
        """
    
    elif OLS_test:
        OLS_mean_weight, FF3_mean_weight, EN_mean_weight = 1, 0, 0
        OLS_cov_weight, FF3_cov_weight, EN_cov_weight = 1, 0, 0
    elif FF3_test:
        OLS_mean_weight, FF3_mean_weight, EN_mean_weight = 0, 1, 0
        OLS_cov_weight, FF3_cov_weight, EN_cov_weight = 0, 1, 0
    elif EN_test:
        OLS_mean_weight, FF3_mean_weight, EN_mean_weight = 0, 0, 1
        OLS_cov_weight, FF3_cov_weight, EN_cov_weight = 0, 0, 1
    else:
        OLS_mean_weight = FF3_mean_weight = EN_mean_weight = 1/3
        OLS_cov_weight = FF3_cov_weight = EN_cov_weight = 1/3

    ensemble_mean = OLS_mean_weight * mean_OLS + FF3_mean_weight * mean_FF3 + EN_mean_weight * mean_EN
    ensemble_cov = OLS_cov_weight * cov_OLS + FF3_cov_weight * cov_FF3 + EN_cov_weight * cov_EN

    alloc_ensemble = MVO(ensemble_mean, ensemble_cov, prev_weights, Transaction_costs)

    return (alloc_ensemble, alpha_OLS, beta_OLS, alpha_FF3, beta_FF3, alpha_EN, beta_EN,
            cov_OLS, cov_FF3, cov_EN, OLS_mean_weight, FF3_mean_weight, EN_mean_weight,
            OLS_cov_weight, FF3_cov_weight, EN_cov_weight)
    
    



In [ ]:
def run_ensemble_backtest(holding_period, training_window, trans_cost_rate, 
                          all_factor_returns, all_excess_returns, Temperature_M,
                          all_abs_returns, initial_value, risk_free_rate,
                          equal_weight, OLS_test, FF3_test, EN_test, lambda_w,
                          lambda_schedule=None):
    """Ensemble backtest. Accepts optional lambda_schedule dict keyed by rebalance index -> (lambda1, lambda2).
    """
    portfolio_value = float(initial_value)

    factor_returns = np.asarray(all_factor_returns, dtype=float)
    excess_returns = np.asarray(all_excess_returns, dtype=float)
    abs_returns = np.asarray(all_abs_returns, dtype=float)

    N = excess_returns.shape[1]
    total_time = factor_returns.shape[0]

    Num_Ext_Weights = 0

    current_weights = np.repeat(1.0 / N, N)

    alpha_OLS_prev = None; alpha_FF3_prev = None; alpha_EN_prev = None
    alpha_OLS_prev_2 = None; alpha_FF3_prev_2 = None; alpha_EN_prev_2 = None
    alpha_OLS_prev_3 = None; alpha_FF3_prev_3 = None; alpha_EN_prev_3 = None

    beta_OLS_prev = None; beta_FF3_prev = None; beta_EN_prev = None
    beta_OLS_prev_2 = None; beta_FF3_prev_2 = None; beta_EN_prev_2 = None
    beta_OLS_prev_3 = None; beta_FF3_prev_3 = None; beta_EN_prev_3 = None

    is_first = True

    portfolio_value_history = [initial_value]
    portfolio_return_history = []
    portfolio_excess_return_history = []
    weight_history = []
    turnover_history = []
    transaction_cost_history = []
    rebalance_history = []
    model_Mweight_history = []
    model_Qweight_history = []

    rebalance_indices = set(range(training_window, total_time, holding_period))

    for time in range(total_time):
        
        if time >= training_window:
            starting_portfolio_value = portfolio_value
        
        if (time in rebalance_indices):
            # fetch lambdas from schedule if provided
            if lambda_schedule is not None:
                opt_lambda_1, opt_lambda_2 = lambda_schedule.get(time, (None, None))
            else:
                opt_lambda_1, opt_lambda_2 = (None, None)

            ( new_weights , alpha_OLS , beta_OLS , alpha_FF3 , beta_FF3 , 
             alpha_EN, beta_EN ,
             cov_OLS , cov_FF3 , cov_EN ,
             OLS_Mweight, FF3_Mweight, EN_Mweight,
             OLS_Qweight, FF3_Qweight, EN_Qweight) = rebalancing(
                        factor_returns[time - training_window:time, :], excess_returns[time - training_window:time, :],
                        abs_returns[time - training_window:time, :],
                        current_weights, trans_cost_rate, Temperature_M,
                        beta_OLS_prev, beta_FF3_prev, beta_EN_prev,
                        alpha_OLS_prev, alpha_FF3_prev, alpha_EN_prev,
                        beta_OLS_prev_2, beta_FF3_prev_2, beta_EN_prev_2,
                        alpha_OLS_prev_2, alpha_FF3_prev_2, alpha_EN_prev_2,
                        beta_OLS_prev_3, beta_FF3_prev_3, beta_EN_prev_3,
                        alpha_OLS_prev_3, alpha_FF3_prev_3, alpha_EN_prev_3,
                        is_first,
                        equal_weight, OLS_test, FF3_test, EN_test, holding_period, lambda_w,
                        opt_lambda_1, opt_lambda_2)
            
            turnover = calculate_turnover(new_weights, current_weights)
            cost_fraction = trans_cost_rate * turnover * 2
            dollar_cost = portfolio_value * cost_fraction
            portfolio_value -= dollar_cost

            current_weights = new_weights.copy()

            turnover_history.append(turnover)
            transaction_cost_history.append(dollar_cost)
            rebalance_history.append(time)

            alpha_OLS_prev_3 = alpha_OLS_prev_2
            beta_OLS_prev_3 = beta_OLS_prev_2
            alpha_FF3_prev_3 = alpha_FF3_prev_2
            beta_FF3_prev_3 = beta_FF3_prev_2
            alpha_EN_prev_3 = alpha_EN_prev_2
            beta_EN_prev_3 = beta_EN_prev_2

            alpha_OLS_prev_2 = alpha_OLS_prev
            beta_OLS_prev_2 = beta_OLS_prev
            alpha_FF3_prev_2 = alpha_FF3_prev
            beta_FF3_prev_2 = beta_FF3_prev
            alpha_EN_prev_2 = alpha_EN_prev
            beta_EN_prev_2 = beta_EN_prev

            alpha_OLS_prev = alpha_OLS
            beta_OLS_prev = beta_OLS
            alpha_FF3_prev = alpha_FF3
            beta_FF3_prev = beta_FF3
            alpha_EN_prev = alpha_EN
            beta_EN_prev = beta_EN

            is_first = False

            model_Mweight_history.append(np.asarray([round(OLS_Mweight,4), round(FF3_Mweight,4), round(EN_Mweight,4)]))
            model_Qweight_history.append(np.asarray([round(OLS_Qweight,4), round(FF3_Qweight,4), round(EN_Qweight,4)]))

            if OLS_Mweight <= 0.1 or FF3_Mweight <= 0.1 or EN_Mweight <= 0.1:
                Num_Ext_Weights += 1

        if time >= training_window:
            current_asset_returns = abs_returns[time, :]

            gross_portfolio_return = float(current_weights @ current_asset_returns)

            portfolio_value *= (1.0 + gross_portfolio_return)

            net_monthly_return = portfolio_value / starting_portfolio_value - 1

            portfolio_return_history.append(net_monthly_return)
            portfolio_value_history.append(portfolio_value)
            
            weight_history.append(np.round(current_weights.copy(),4))
            current_weights = drift_weights(current_weights, current_asset_returns)
            
    average_turnover = np.mean(turnover_history) if len(turnover_history)>0 else 0.0

    # Ensure the risk_free_rate slicing aligns with portfolio_return_history length
    rf_slice = np.asarray(risk_free_rate.iloc[training_window:training_window + len(portfolio_return_history)])
    portfolio_excess_return_history = np.asarray(portfolio_return_history) - rf_slice

    sharpe_ratio = ( np.sqrt(12) * (np.mean(portfolio_excess_return_history) )
                        / (np.std(portfolio_excess_return_history, ddof=1) if len(portfolio_excess_return_history)>1 else np.nan) )
        
    portfolio_return_history_annualized = (np.asarray(1 + np.asarray(portfolio_return_history)) ** 12 - 1
                                           - (rf_slice + 1) ** 12 + 1)

    mean_return = np.mean(portfolio_return_history_annualized) if len(portfolio_return_history_annualized)>0 else np.nan
    std_return = np.std(portfolio_return_history_annualized, ddof=1) if len(portfolio_return_history_annualized)>1 else np.nan

    return {"final_portfolio_value": portfolio_value,
            "portfolio_value_history": np.asarray(portfolio_value_history),
            "portfolio_return_history_annualized": portfolio_return_history_annualized,
            "weight_history": np.asarray(weight_history),
            "turnover_history": np.asarray(turnover_history),
            "average_turnover": average_turnover,
            "transaction_cost_history": np.asarray(transaction_cost_history),
            "rebalance_indices": rebalance_history,
            "sharpe_ratio": sharpe_ratio,
            "model_Mweight_history": model_Mweight_history, 
            "model_Qweight_history": model_Qweight_history,
            "portfolio_return_history": np.asarray(portfolio_return_history), 
            "mean_return": mean_return, "std_return": std_return,
            "Num_Ext_Weights": Num_Ext_Weights}

In [ ]:
holding_period = 6
# total_time = 120
initial_value = 100000
training_window = 60
lambda_w = 0.75

In [ ]:
def plot_metric_heatmap_regular_temp_reversed(
    metric_values,temperatures,title,colorbar_label,decimals=3):
    
    fig, ax = plt.subplots(figsize=(4, 7))

    from matplotlib.colors import TwoSlopeNorm
    import numpy as np

    metric_values = np.asarray(metric_values).flatten()

    # Temperature = 0 baseline
    baseline_idx = np.where(np.isclose(temperatures, 0))[0][0]
    baseline = metric_values[baseline_idx]

    # Make colour scale symmetric around the baseline
    max_deviation = np.max(np.abs(metric_values - baseline))

    norm = TwoSlopeNorm(vmin=baseline - max_deviation,
        vcenter=baseline,vmax=baseline + max_deviation)

    # Convert to one-column grid
    metric_grid = metric_values.reshape(-1, 1)

    image = ax.imshow(
        metric_grid,aspect="auto",origin="lower",cmap="RdYlGn_r",norm=norm)

    # Y-axis
    ax.set_yticks(np.arange(len(temperatures)))
    ax.set_yticklabels(temperatures)
    ax.set_ylabel("Softmax Temperature (Selectivity)")

    # Single column
    ax.set_xticks([0])
    ax.set_xticklabels(["Aggregated"])
    ax.set_title(title)

    colorbar = fig.colorbar(image, ax=ax)
    colorbar.set_label(colorbar_label)

    # Write metric value inside each cell
    for i in range(len(metric_values)):
        value = metric_values[i]

        ax.text(0,i,f"{value:.{decimals}f}",ha="center",va="center")

    plt.tight_layout()
    plt.show()

In [ ]:
# Define a function to plot a heatmap of metric values against temperatures, 
# with a regular color scale (not reversed)- will be used for plotting Sharpe ratio 
# and mean return heatmaps when comparing different base models vs. ensemble model

def plot_metric_heatmap_regular_temp(
    metric_values,temperatures,title,colorbar_label,decimals=3):

    fig, ax = plt.subplots(figsize=(4, 7))

    # Get preferred colormap for regular heatmap
    from matplotlib.colors import TwoSlopeNorm
    import numpy as np

    metric_values = np.asarray(metric_values).flatten()

    # Temperature = 0 baseline
    baseline_idx = np.where(np.isclose(temperatures, 0))[0][0]
    baseline = metric_values[baseline_idx]

    # Make colour scale symmetric around the baseline
    max_deviation = np.max(np.abs(metric_values - baseline))

    norm = TwoSlopeNorm(vmin=baseline - max_deviation,
        vcenter=baseline,vmax=baseline + max_deviation)

    # Convert to one-column grid
    metric_grid = metric_values.reshape(-1, 1)

    image = ax.imshow(
        metric_grid,aspect="auto",origin="lower",cmap="RdYlGn",norm=norm)

    # Y-axis
    ax.set_yticks(np.arange(len(temperatures)))
    ax.set_yticklabels(temperatures)
    ax.set_ylabel("Softmax Temperature (Selectivity)")

    # Single column
    ax.set_xticks([0])
    ax.set_xticklabels(["Aggregated"])
    ax.set_title(title)

    colorbar = fig.colorbar(image, ax=ax)
    colorbar.set_label(colorbar_label)

    # Write metric value inside each cell- used for Appendix graphs
    for i in range(len(metric_values)):
        value = metric_values[i]

        ax.text(0,i,f"{value:.{decimals}f}",
            ha="center",va="center")

    plt.tight_layout()
    plt.show()

In [ ]:

# Define a function to plot a heatmap of metric values against temperatures and transaction costs,
# with a reversed color scale (green = better, red = worse) - will be used
# for plotting turnover heatmaps for ensemble on its own

def plot_metric_heatmap_reversed_abs(metric_grid,temperatures,transaction_costs_bps,
    title,colorbar_label,decimals=3):

    fig, ax = plt.subplots(figsize=(10, 7))

    image = ax.imshow(
        metric_grid,aspect="auto",origin="lower",cmap="RdYlGn_r")

    ax.set_xticks(np.arange(len(transaction_costs_bps)))
    ax.set_xticklabels(transaction_costs_bps)
    ax.set_yticks(np.arange(len(temperatures)))
    ax.set_yticklabels(temperatures)
    ax.set_xlabel("Transaction Cost (bps)")
    ax.set_ylabel("Softmax Temperature (Selectivity)")
    ax.set_title(title)

    colorbar = fig.colorbar(image,ax=ax)
    colorbar.set_label(colorbar_label)

    # Write value inside each cell- used for Appendix graphs
    for i in range(metric_grid.shape[0]):
        for j in range(metric_grid.shape[1]):
            value = metric_grid[i, j]
            ax.text(j,i,
                f"{value:.{decimals}f}",
                ha="center",
                va="center")
    

    plt.tight_layout()
    plt.show()

In [ ]:
# Define a function to plot a heatmap of metric values against temperatures and transaction costs,
# with a reversed color scale (green = better, red = worse) - will be used
# for plotting Sharpe heatmaps for the ensemble on its own

def plot_metric_heatmap_regular_abs(metric_grid,temperatures,transaction_costs_bps,
    title,colorbar_label,decimals=3):
    
    fig, ax = plt.subplots(figsize=(10, 7))

    image = ax.imshow(
        metric_grid,aspect="auto",origin="lower",cmap="RdYlGn")

    ax.set_xticks(np.arange(len(transaction_costs_bps)))
    ax.set_xticklabels(transaction_costs_bps)
    ax.set_yticks(np.arange(len(temperatures)))
    ax.set_yticklabels(temperatures)
    ax.set_xlabel("Transaction Cost (bps)")
    ax.set_ylabel("Softmax Temperature (Selectivity)")
    ax.set_title(title)

    colorbar = fig.colorbar(image,ax=ax)
    colorbar.set_label(colorbar_label)

    # Write value inside each cell- used for Appendix graphs
    for i in range(metric_grid.shape[0]):
        for j in range(metric_grid.shape[1]):
            value = metric_grid[i, j]
            ax.text(j,i,
                f"{value:.{decimals}f}",
                ha="center",
                va="center")
    

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_metric_heatmap_regular(metric_grid,temperatures,transaction_costs_bps,
    title,colorbar_label,decimals=3):
    
    fig, ax = plt.subplots(figsize=(10, 7))

    from matplotlib.colors import TwoSlopeNorm
    max_abs = np.max(np.abs(metric_grid))

    norm = TwoSlopeNorm(vmin=-max_abs,vcenter=0,vmax=max_abs)

    image = ax.imshow(
        metric_grid,aspect="auto",origin="lower",cmap="RdYlGn",norm=norm)

    ax.set_xticks(np.arange(len(transaction_costs_bps)))
    ax.set_xticklabels(transaction_costs_bps)
    ax.set_yticks(np.arange(len(temperatures)))
    ax.set_yticklabels(temperatures)
    ax.set_xlabel("Transaction Cost (bps)")
    ax.set_ylabel("Softmax Temperature (Selectivity)")
    ax.set_title(title)

    colorbar = fig.colorbar(image,ax=ax)
    colorbar.set_label(colorbar_label)

    # Write value inside each cell- used for Appendix graphs
    for i in range(metric_grid.shape[0]):
        for j in range(metric_grid.shape[1]):
            value = metric_grid[i, j]
            ax.text(j,i,
                f"{value:.{decimals}f}",
                ha="center",
                va="center")
    
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_metric_heatmap_reversed(metric_grid,temperatures,transaction_costs_bps,
    title,colorbar_label,decimals=3):
    
    fig, ax = plt.subplots(figsize=(10, 7))

    from matplotlib.colors import TwoSlopeNorm
    max_abs = np.max(np.abs(metric_grid))

    norm = TwoSlopeNorm(
        vmin=-max_abs,vcenter=0,vmax=max_abs)

    image = ax.imshow(
        metric_grid,aspect="auto",origin="lower",cmap="RdYlGn_r",norm=norm)

    ax.set_xticks(np.arange(len(transaction_costs_bps)))
    ax.set_xticklabels(transaction_costs_bps)
    ax.set_yticks(np.arange(len(temperatures)))
    ax.set_yticklabels(temperatures)
    ax.set_xlabel("Transaction Cost (bps)")
    ax.set_ylabel("Softmax Temperature (Selectivity)")
    ax.set_title(title)

    colorbar = fig.colorbar(image,ax=ax)
    colorbar.set_label(colorbar_label)

    # Write value inside each cell- used for Appendix graphs
    for i in range(metric_grid.shape[0]):
        for j in range(metric_grid.shape[1]):
            value = metric_grid[i, j]
            ax.text(j,i,
                f"{value:.{decimals}f}",
                ha="center",
                va="center")
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Create Sharpe and Turnover Grid

# Manually set the number of temperatures and transaction costs to match the grid size
num_temps = 31
num_trans = 11

# Choose dataset 1 for the first backtest
all_factor_returns_1 = factor_returns_d1
all_excess_returns_1 = excess_returns_d1
all_abs_returns_1 = monthly_returns_d1

risk_free_rate_1 = risk_free_rate_d1

# Manually set the the array of temperatures and transactions costs
Temperatures =  np.array([0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.50, 
                          0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0,
                          1.05, 1.1, 1.15, 1.2, 1.25, 1.3, 1.35, 1.4, 1.45, 1.5,])
Transaction_Costs_bps = np.arange(0,51,5)
Transaction_Costs = Transaction_Costs_bps / 10000

# Initialize results arrays for Sharpe ratio, Turnover, Number of Extreme Weights, Mean Return, and Standard Deviation of Return
Sharpe_grid1 = np.zeros((num_temps, num_trans))
Turnover_grid1 = np.zeros((num_temps, num_trans))
Num_Ext_grid1 = np.zeros((num_temps, num_trans))
Mean_return_grid1 = np.zeros((num_temps, num_trans))
std_return_grid1 = np.zeros((num_temps, num_trans))


In [ ]:
# Precompute the lambda schedule for the entire dataset to avoid redundant calculations during the grid search
# Note that this results in 341x fewer calls to get_lambdas, which is a significant time savings

# Ensure that the factor returns and excess returns are numpy arrays
factor_returns = np.asarray(all_factor_returns_1,dtype=float)
excess_returns = np.asarray(all_excess_returns_1,dtype=float)

rebalance_indices = list(range(training_window,factor_returns.shape[0],
        holding_period))

# Initialize a dictionary to store the lambda values for each rebalance index
lambda_schedule = {}

# Compute the lambdas at each rebalancing point
for time in rebalance_indices:
    train_factor_returns = factor_returns[time - training_window:time,:]
    train_excess_returns = excess_returns[time - training_window:time,:]

    lambda_1, lambda_2, _ = get_lambdas(
        train_excess_returns,train_factor_returns)

    lambda_schedule[time] = (lambda_1,lambda_2)

    # Print the computed lambda values
    print("time:",time,"lambda1:",lambda_1,"lambda2:",lambda_2)

In [ ]:
# Backtest the ensemble model across the grid of temperatures and transaction costs, using the precomputed lambda schedule

for i, temp in enumerate(Temperatures):
    for j, trans_cost in enumerate(Transaction_Costs):
        # Run the ensemble backtest for the given temperature and transaction cost
        # Use the precomputed lambda schedule to avoid redundant calculations
        # Not using equal_weight, OLS_test, FF3_test, or EN_test indicators for this backtest
        results1 = run_ensemble_backtest(
            holding_period, training_window, Transaction_Costs[j],
                      all_factor_returns_1, all_excess_returns_1, 
                      Temperatures[i],
                      all_abs_returns_1, initial_value, risk_free_rate_1,
                      False, False, False, False, lambda_w, lambda_schedule)
        print(results1["model_Mweight_history"])
        # Fill in the Sharpe and Turnover grids with the results
        Sharpe_grid1[i,j] = results1["sharpe_ratio"]
        Turnover_grid1[i,j] = results1["average_turnover"]
        Num_Ext_grid1[i,j] = results1["Num_Ext_Weights"]
        Mean_return_grid1[i,j] = results1["mean_return"]
        std_return_grid1[i,j] = results1["std_return"]


In [ ]:
# Run the backtest for the OLS model across the grid of temperatures and transaction costs, using the precomputed lambda schedule

# Manually set the array of transaction costs
Transaction_Costs_bps = np.arange(0,51,5)
Transaction_Costs = Transaction_Costs_bps / 10000
Sharpe_grid_OLS1 = np.zeros((num_temps, num_trans))
Turnover_grid_OLS1 = np.zeros((num_temps, num_trans))
Num_Ext_grid_OLS1 = np.zeros((num_temps, num_trans))
Mean_return_grid_OLS1 = np.zeros((num_temps, num_trans))
std_return_grid_OLS1 = np.zeros((num_temps, num_trans))

# Using the OLS_test indicator to run the backtest for the OLS model only
for j, trans_cost in enumerate(Transaction_Costs):
    results_OLS1 = run_ensemble_backtest(holding_period, training_window, Transaction_Costs[j],
                                    all_factor_returns_1, all_excess_returns_1, 
                      0,
                      all_abs_returns_1, initial_value, risk_free_rate_1,
                      False, True, False, False, lambda_w, lambda_schedule)
    for i, temp in enumerate(Temperatures):
        Sharpe_grid_OLS1[i][j] = results_OLS1["sharpe_ratio"]
        Turnover_grid_OLS1[i][j] = results_OLS1["average_turnover"]
        Num_Ext_grid_OLS1[i][j] = results_OLS1["Num_Ext_Weights"]
        Mean_return_grid_OLS1[i][j] = results_OLS1["mean_return"]
        std_return_grid_OLS1[i][j] = results_OLS1["std_return"]


In [ ]:
# Run the backtest for the FF3 model across the grid of temperatures and transaction costs, using the precomputed lambda schedule

# Manually set the array of transaction costs
Transaction_Costs_bps = np.arange(0,51,5)
Transaction_Costs = Transaction_Costs_bps / 10000
Sharpe_grid_FF31 = np.zeros((num_temps, num_trans))
Turnover_grid_FF31 = np.zeros((num_temps, num_trans))
Num_Ext_grid_FF31 = np.zeros((num_temps, num_trans))
Mean_return_grid_FF31 = np.zeros((num_temps, num_trans))
std_return_grid_FF31 = np.zeros((num_temps, num_trans))

# Using the FF3_test indicator to run the backtest for the FF3 model only
for j, trans_cost in enumerate(Transaction_Costs):
    results_FF31 = run_ensemble_backtest(holding_period, training_window, Transaction_Costs[j],
                                    all_factor_returns_1, all_excess_returns_1, 
                      0,
                      all_abs_returns_1, initial_value, risk_free_rate_1,
                      False, False, True, False, lambda_w, lambda_schedule)
    for i, temp in enumerate(Temperatures):
        Sharpe_grid_FF31[i][j] = results_FF31["sharpe_ratio"]
        Turnover_grid_FF31[i][j] = results_FF31["average_turnover"]
        Num_Ext_grid_FF31[i][j] = results_FF31["Num_Ext_Weights"]
        Mean_return_grid_FF31[i][j] = results_FF31["mean_return"]
        std_return_grid_FF31[i][j] = results_FF31["std_return"]

In [ ]:
# Run the backtest for the EN model across the grid of temperatures and transaction costs, using the precomputed lambda schedule

# Manually set the the array of transactions costs
Transaction_Costs_bps = np.arange(0,51,5)
Transaction_Costs = Transaction_Costs_bps / 10000
Sharpe_grid_EN1 = np.zeros((num_temps, num_trans))
Turnover_grid_EN1 = np.zeros((num_temps, num_trans))
Num_Ext_grid_EN1 = np.zeros((num_temps, num_trans))
Mean_return_grid_EN1 = np.zeros((num_temps, num_trans))
std_return_grid_EN1 = np.zeros((num_temps, num_trans))

# Using the EN_test indicator to run the backtest for the EN model only
for j, trans_cost in enumerate(Transaction_Costs):
    results_EN1 = run_ensemble_backtest(holding_period, training_window, Transaction_Costs[j],
                                    all_factor_returns_1, all_excess_returns_1, 
                      0,
                      all_abs_returns_1, initial_value, risk_free_rate_1,
                      False, False, False, True, lambda_w, lambda_schedule)
    for i, temp in enumerate(Temperatures):
        Sharpe_grid_EN1[i][j] = results_EN1["sharpe_ratio"]
        Turnover_grid_EN1[i][j] = results_EN1["average_turnover"]
        Num_Ext_grid_EN1[i][j] = results_EN1["Num_Ext_Weights"]
        Mean_return_grid_EN1[i][j] = results_EN1["mean_return"]
        std_return_grid_EN1[i][j] = results_EN1["std_return"]

In [ ]:
# Plot the relative Sharpe performance for ensemble models against OLS
plot_metric_heatmap_regular(Sharpe_grid1 - Sharpe_grid_OLS1,Temperatures, Transaction_Costs_bps,
    "Sharpe Performance Difference: Ensemble vs. OLS - Dataset 1","Sharpe Difference",decimals=3)

In [ ]:
# Plot the relative Sharpe performance for ensemble models against FF3
plot_metric_heatmap_regular(Sharpe_grid1 - Sharpe_grid_FF31,Temperatures,Transaction_Costs_bps,
    "Sharpe Ratio Ensemble vs FF3 - Dataset 1","Sharpe",decimals=3)

In [ ]:
# Plot the absolute Sharpe performance for ensemble models
plot_metric_heatmap_regular_abs(Sharpe_grid1,Temperatures,Transaction_Costs_bps,
    "Sharpe Ratio: Ensemble - Dataset 1","Sharpe",decimals=3)

In [ ]:
# Plot the relative Sharpe performance for ensemble models against EN
plot_metric_heatmap_regular(Sharpe_grid1 - Sharpe_grid_EN1,Temperatures,Transaction_Costs_bps,
    "Sharpe Ratio: Ensemble vs EN - Dataset 1","Sharpe Difference",decimals=3)

In [ ]:
# Plot the absolute Turnover performance for ensemble models
plot_metric_heatmap_reversed_abs(Turnover_grid1,Temperatures,Transaction_Costs_bps,
    "Average Portfolio Turnover Ensemble - Dataset 1","Average Turnover",decimals=3)

In [ ]:
# Plot the relative Turnover performance for ensemble models against OLS
plot_metric_heatmap_reversed(Turnover_grid1 - Turnover_grid_OLS1,Temperatures,
    Transaction_Costs_bps,"Average Portfolio Turnover Ensemble vs. OLS - Dataset 1","Average Turnover",decimals=3)

In [ ]:
# Plot the relative Turnover performance for ensemble models aginst FF3
plot_metric_heatmap_reversed(Turnover_grid1 - Turnover_grid_FF31,Temperatures,Transaction_Costs_bps,
    "Average Portfolio Turnover Ensemble vs. FF3 - Dataset 1","Average Turnover",decimals=3)

In [ ]:
# Plot the relative Turnover performance for ensemble models against EN
plot_metric_heatmap_reversed(Turnover_grid1 - Turnover_grid_EN1,Temperatures,Transaction_Costs_bps,
    "Average Portfolio Turnover Ensemble vs. EN - Dataset 1","Average Turnover",decimals=3)

In [ ]:
        
# Create Sharpe and Turnover Grid
num_temps = 31
num_trans = 11

# Choose dataset 2 for the next backtest
all_factor_returns_2 = factor_returns_d2
all_excess_returns_2 = excess_returns_d2
all_abs_returns_2 = monthly_returns_d2

risk_free_rate_2 = risk_free_rate_d2

# Manually set temperatures and transaction costs for the grid search
Temperatures =  np.array([0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.50, 
                          0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0, 
                          1.05, 1.1, 1.15, 1.2, 1.25, 1.3, 1.35, 1.4, 1.45, 1.5])
Transaction_Costs_bps = np.arange(0,51,5)
Transaction_Costs = Transaction_Costs_bps / 10000
Sharpe_grid2 = np.zeros((num_temps, num_trans))
Turnover_grid2 = np.zeros((num_temps, num_trans))
Num_Ext_grid2 = np.zeros((num_temps, num_trans))
Mean_return_grid2 = np.zeros((num_temps, num_trans))
std_return_grid2 = np.zeros((num_temps, num_trans))


In [ ]:
# Precompute the lambda schedule for the entire dataset 2 to avoid 
# redundant calculations during the grid search

# Ensure that the factor returns and excess returns are numpy arrays
factor_returns = np.asarray(all_factor_returns_2,dtype=float)
excess_returns = np.asarray(all_excess_returns_2,dtype=float)

rebalance_indices = list(
    range(training_window,factor_returns.shape[0],holding_period))

# Initialize a dictionary to store the lambda values for each rebalance index
lambda_schedule_2 = {}

# Compute the lambdas at each rebalancing point
for time in rebalance_indices:

    train_factor_returns = factor_returns[time - training_window:time,:]
    train_excess_returns = excess_returns[time - training_window:time,:]

    lambda_1, lambda_2, _ = get_lambdas(train_excess_returns,train_factor_returns)

    lambda_schedule_2[time] = (lambda_1,lambda_2)

    # Print out the lambda values for each rebalance index for the second dataset
    print("time:",time,"lambda1:",lambda_1,"lambda2:",lambda_2)

In [ ]:
# Run the backtest for the ensemble models across 
# the grid of temperatures and transaction costs for dataset 2

for i, temp in enumerate(Temperatures):
    for j, trans_cost in enumerate(Transaction_Costs):
        # Run the ensemble backtest for the given temperature and transaction cost
        results2 = run_ensemble_backtest(
            holding_period, training_window, Transaction_Costs[j],
                      all_factor_returns_2, all_excess_returns_2, 
                      Temperatures[i],
                      all_abs_returns_2, initial_value, risk_free_rate_2,
                      False, False, False, False, lambda_w, lambda_schedule_2)
        print(results2["model_Mweight_history"])
        
        # Fill in the Sharpe and Turnover grids with the results
        Sharpe_grid2[i,j] = results2["sharpe_ratio"]
        Turnover_grid2[i,j] = results2["average_turnover"]
        Num_Ext_grid2[i,j] = results2["Num_Ext_Weights"]
        Mean_return_grid2[i,j] = results2["mean_return"]
        std_return_grid2[i,j] = results2["std_return"]


In [ ]:
# Manually set the array of transaction costs
Transaction_Costs_bps = np.arange(0,51,5)
Transaction_Costs = Transaction_Costs_bps / 10000
Sharpe_grid_OLS2 = np.zeros((num_temps, num_trans))
Turnover_grid_OLS2 = np.zeros((num_temps, num_trans))
Num_Ext_grid_OLS2 = np.zeros((num_temps, num_trans))
Mean_return_grid_OLS2 = np.zeros((num_temps, num_trans))
std_return_grid_OLS2 = np.zeros((num_temps, num_trans))

# Run the backtest model for the OLS model, using the OLS_test indicator to run the backtest 
for j, trans_cost in enumerate(Transaction_Costs):
    results_OLS2 = run_ensemble_backtest(holding_period, training_window, Transaction_Costs[j],
                                    all_factor_returns_2, all_excess_returns_2, 
                      0,
                      all_abs_returns_2, initial_value, risk_free_rate_2,
                      False, True, False, False, lambda_w, lambda_schedule_2)
    for i, temp in enumerate(Temperatures):
        Sharpe_grid_OLS2[i][j] = results_OLS2["sharpe_ratio"]
        Turnover_grid_OLS2[i][j] = results_OLS2["average_turnover"]
        Num_Ext_grid_OLS2[i][j] = results_OLS2["Num_Ext_Weights"]
        Mean_return_grid_OLS2[i][j] = results_OLS2["mean_return"]
        std_return_grid_OLS2[i][j] = results_OLS2["std_return"]


In [ ]:
# Manually set the array of transaction costs
Transaction_Costs_bps = np.arange(0,51,5)
Transaction_Costs = Transaction_Costs_bps / 10000
Sharpe_grid_FF32 = np.zeros((num_temps, num_trans))
Turnover_grid_FF32 = np.zeros((num_temps, num_trans))
Num_Ext_grid_FF32 = np.zeros((num_temps, num_trans))
Mean_return_grid_FF32 = np.zeros((num_temps, num_trans))
std_return_grid_FF32 = np.zeros((num_temps, num_trans))

# Run the backtest for the FF3 model, using the FF3_test indicator
for j, trans_cost in enumerate(Transaction_Costs):
    results_FF32 = run_ensemble_backtest(holding_period, training_window, Transaction_Costs[j],
                                    all_factor_returns_2, all_excess_returns_2, 
                      0,
                      all_abs_returns_2, initial_value, risk_free_rate_2,
                      False, False, True, False, lambda_w, lambda_schedule_2)
    for i, temp in enumerate(Temperatures):
        Sharpe_grid_FF32[i][j] = results_FF32["sharpe_ratio"]
        Turnover_grid_FF32[i][j] = results_FF32["average_turnover"]
        Num_Ext_grid_FF32[i][j] = results_FF32["Num_Ext_Weights"]
        Mean_return_grid_FF32[i][j] = results_FF32["mean_return"]
        std_return_grid_FF32[i][j] = results_FF32["std_return"]

In [ ]:
# Now the backtest for the EN model, using the EN_test indicator

# Manually set the array of transaction costs
Transaction_Costs_bps = np.arange(0,51,5)
Transaction_Costs = Transaction_Costs_bps / 10000
Sharpe_grid_EN2 = np.zeros((num_temps, num_trans))
Turnover_grid_EN2 = np.zeros((num_temps, num_trans))
Num_Ext_grid_EN2 = np.zeros((num_temps, num_trans))
Mean_return_grid_EN2 = np.zeros((num_temps, num_trans))
std_return_grid_EN2 = np.zeros((num_temps, num_trans))

# Using the EN_test flag
for j, trans_cost in enumerate(Transaction_Costs):
    results_EN2 = run_ensemble_backtest(holding_period, training_window, Transaction_Costs[j],
                                    all_factor_returns_2, all_excess_returns_2, 
                      0,
                      all_abs_returns_2, initial_value, risk_free_rate_2,
                      False, False, False, True, lambda_w, lambda_schedule_2)
    for i, temp in enumerate(Temperatures):
        Sharpe_grid_EN2[i][j] = results_EN2["sharpe_ratio"]
        Turnover_grid_EN2[i][j] = results_EN2["average_turnover"]
        Num_Ext_grid_EN2[i][j] = results_EN2["Num_Ext_Weights"]
        Mean_return_grid_EN2[i][j] = results_EN2["mean_return"]
        std_return_grid_EN2[i][j] = results_EN2["std_return"]

In [ ]:
# Plot the absolute Sharpe performance for ensemble models for dataset 2
plot_metric_heatmap_regular_abs(Sharpe_grid2, Temperatures, Transaction_Costs_bps,
    "Sharpe Performance: Ensemble - Dataset 2", "Sharpe Difference", decimals=3)

In [ ]:
# Plot the relative Sharpe performance for ensemble models against OLS for dataset 2
plot_metric_heatmap_regular(Sharpe_grid2 - Sharpe_grid_OLS2, Temperatures, Transaction_Costs_bps,
    "Sharpe Performance: Ensemble vs. OLS - Dataset 2", "Sharpe Difference", decimals=3)

In [ ]:
# Plot the relative Sharpe performance for ensemble models against FF3 for dataset 2
plot_metric_heatmap_regular(Sharpe_grid2 - Sharpe_grid_FF32, Temperatures, Transaction_Costs_bps,
    "Sharpe Performance: Ensemble vs. FF3 - Dataset 2", "Sharpe Difference", decimals=3)

In [ ]:
# Plot the relative Sharpe performance for ensemble models against EN for dataset 2
plot_metric_heatmap_regular(Sharpe_grid2 - Sharpe_grid_EN2, Temperatures, Transaction_Costs_bps,
    "Sharpe Performance: Ensemble vs. EN - Dataset 2", "Sharpe Difference", decimals=3)

In [ ]:
# Plot the absolute Turnover performance for ensemble models for dataset 2
plot_metric_heatmap_reversed_abs(Turnover_grid2, Temperatures, Transaction_Costs_bps,
    "Average Portfolio Turnover Ensemble - Dataset 2", "Average Turnover", decimals=3)

In [ ]:
# Plot the relative Turnover performance for ensemble models against OLS for dataset 2
plot_metric_heatmap_reversed(Turnover_grid2 - Turnover_grid_OLS2, Temperatures, Transaction_Costs_bps,
    "Average Portfolio Turnover Ensemble vs. OLS - Dataset 2", "Average Turnover", decimals=3)

In [ ]:
# Plot the relative Turnover performance for ensemble models against FF3 for dataset 2
plot_metric_heatmap_reversed(Turnover_grid2 - Turnover_grid_FF32, Temperatures, Transaction_Costs_bps,
    "Average Portfolio Turnover Ensemble vs. FF3 - Dataset 2", "Average Turnover", decimals=3)

In [ ]:
# Plot the relative Turnover performance for ensemble models against EN for dataset 2
plot_metric_heatmap_reversed(Turnover_grid2 - Turnover_grid_EN2, Temperatures, Transaction_Costs_bps,
    "Average Portfolio Turnover Ensemble vs. EN - Dataset 2", "Average Turnover", decimals=3)

In [ ]:
# Create Sharpe and Turnover Grid
# Have manually set the number of temperatures and transaction costs to match the grid size
num_temps = 31
num_trans = 11

# Choose dataset 3 for the next backtest
all_factor_returns_3 = factor_returns_d3
all_excess_returns_3 = excess_returns_d3
all_abs_returns_3 = monthly_returns_d3

risk_free_rate_3 = risk_free_rate_d3

# Manually set temperatures and transaction costs for the grid search
Temperatures =  np.array([0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.50, 
                          0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0, 
                          1.05, 1.1, 1.15, 1.2, 1.25, 1.3, 1.35, 1.4, 1.45, 1.5])
Transaction_Costs_bps = np.arange(0,51,5)
Transaction_Costs = Transaction_Costs_bps / 10000
Sharpe_grid3 = np.zeros((num_temps, num_trans))
Turnover_grid3 = np.zeros((num_temps, num_trans))
Num_Ext_grid3 = np.zeros((num_temps, num_trans))
Mean_return_grid3 = np.zeros((num_temps, num_trans))
std_return_grid3 = np.zeros((num_temps, num_trans))

In [ ]:
# Precompute the lambda schedule for the entire dataset to avoid redundant calculations during the grid search
# for dataset 3

# Ensure that the factor returns and excess returns are numpy arrays
factor_returns = np.asarray(all_factor_returns_3, dtype=float)
excess_returns = np.asarray(all_excess_returns_3, dtype=float)

rebalance_indices = list(range(training_window,factor_returns.shape[0],
                               holding_period))

# Initialize a dictionary to store the lambda values
lambda_schedule_3 = {}

for time in rebalance_indices:
    train_factor_returns = factor_returns[time - training_window:time,:]
    train_excess_returns = excess_returns[time - training_window:time,:]

    lambda_1, lambda_2, _ = get_lambdas(train_excess_returns,
                                        train_factor_returns)

    lambda_schedule_3[time] = (lambda_1,lambda_2)

    # Print out the lambda values for each rebalance index
    print("time:",time,"lambda1:",lambda_1,"lambda2:",lambda_2)

In [ ]:
for i, temp in enumerate(Temperatures):
    for j, trans_cost in enumerate(Transaction_Costs):
        # Run the ensemble backtest for the given temperature and transaction cost for dataset 3
        # Not using equal_weight, OLS_test, FF3_test, or EN_test indicators for this backtest
        # Running for ensemble model only across different temperatures
        results3 = run_ensemble_backtest(
            holding_period, training_window, Transaction_Costs[j],
                      all_factor_returns_3, all_excess_returns_3, 
                      Temperatures[i],
                      all_abs_returns_3, initial_value, risk_free_rate_3,
                      False, False, False, False, lambda_w, lambda_schedule_3)
        print(results3["model_Mweight_history"])
        # Fill in the Sharpe and Turnover grids with the results
        Sharpe_grid3[i,j] = results3["sharpe_ratio"]
        Turnover_grid3[i,j] = results3["average_turnover"]
        Num_Ext_grid3[i,j] = results3["Num_Ext_Weights"]
        Mean_return_grid3[i,j] = results3["mean_return"]
        std_return_grid3[i,j] = results3["std_return"]

In [ ]:
# 

# Manually set the array of transaction costs for dataset 3
Transaction_Costs_bps = np.arange(0,51,5)
Transaction_Costs = Transaction_Costs_bps / 10000
Sharpe_grid_OLS3 = np.zeros((num_temps, num_trans))
Turnover_grid_OLS3 = np.zeros((num_temps, num_trans))
Num_Ext_grid_OLS3 = np.zeros((num_temps, num_trans))
Mean_return_grid_OLS3 = np.zeros((num_temps, num_trans))
std_return_grid_OLS3 = np.zeros((num_temps, num_trans))

for j, trans_cost in enumerate(Transaction_Costs):
    results_OLS3 = run_ensemble_backtest(holding_period, training_window, Transaction_Costs[j],
                                    all_factor_returns_3, all_excess_returns_3, 
                      0,
                      all_abs_returns_3, initial_value, risk_free_rate_3,
                      False, True, False, False, lambda_w, lambda_schedule_3)
    for i, temp in enumerate(Temperatures):
        Sharpe_grid_OLS3[i][j] = results_OLS3["sharpe_ratio"]
        Turnover_grid_OLS3[i][j] = results_OLS3["average_turnover"]
        Num_Ext_grid_OLS3[i][j] = results_OLS3["Num_Ext_Weights"]
        Mean_return_grid_OLS3[i][j] = results_OLS3["mean_return"]
        std_return_grid_OLS3[i][j] = results_OLS3["std_return"]

Target adjusted


In [367]:
Transaction_Costs_bps = np.arange(0,51,5)
Transaction_Costs = Transaction_Costs_bps / 10000
Sharpe_grid_FF33 = np.zeros((num_temps, num_trans))
Turnover_grid_FF33 = np.zeros((num_temps, num_trans))
Num_Ext_grid_FF33 = np.zeros((num_temps, num_trans))
Mean_return_grid_FF33 = np.zeros((num_temps, num_trans))
std_return_grid_FF33 = np.zeros((num_temps, num_trans))

for j, trans_cost in enumerate(Transaction_Costs):
    results_FF33 = run_ensemble_backtest(holding_period, training_window, Transaction_Costs[j],
                                    all_factor_returns_3, all_excess_returns_3, 
                      0,
                      all_abs_returns_3, initial_value, risk_free_rate_3,
                      False, False, True, False, lambda_w, lambda_schedule_3)
    for i, temp in enumerate(Temperatures):
        Sharpe_grid_FF33[i][j] = results_FF33["sharpe_ratio"]
        Turnover_grid_FF33[i][j] = results_FF33["average_turnover"]
        Num_Ext_grid_FF33[i][j] = results_FF33["Num_Ext_Weights"]
        Mean_return_grid_FF33[i][j] = results_FF33["mean_return"]
        std_return_grid_FF33[i][j] = results_FF33["std_return"]

In [368]:
Transaction_Costs_bps = np.arange(0,51,5)
Transaction_Costs = Transaction_Costs_bps / 10000
Sharpe_grid_EN3 = np.zeros((num_temps, num_trans))
Turnover_grid_EN3 = np.zeros((num_temps, num_trans))
Num_Ext_grid_EN3 = np.zeros((num_temps, num_trans))
Mean_return_grid_EN3 = np.zeros((num_temps, num_trans))
std_return_grid_EN3 = np.zeros((num_temps, num_trans))

for j, trans_cost in enumerate(Transaction_Costs):
    results_EN3 = run_ensemble_backtest(holding_period, training_window, Transaction_Costs[j],
                                    all_factor_returns_3, all_excess_returns_3, 
                      0,
                      all_abs_returns_3, initial_value, risk_free_rate_3,
                      False, False, False, True, lambda_w, lambda_schedule_3)
    for i, temp in enumerate(Temperatures):
        Sharpe_grid_EN3[i][j] = results_EN3["sharpe_ratio"]
        Turnover_grid_EN3[i][j] = results_EN3["average_turnover"]
        Num_Ext_grid_EN3[i][j] = results_EN3["Num_Ext_Weights"]
        Mean_return_grid_EN3[i][j] = results_EN3["mean_return"]
        std_return_grid_EN3[i][j] = results_EN3["std_return"]

Target adjusted


In [ ]:
# Plot the absolute Sharpe performance for ensemble models for dataset 3
plot_metric_heatmap_regular_abs(Sharpe_grid3,Temperatures, Transaction_Costs_bps,
    "Sharpe Performance: Ensemble - Dataset 3","Sharpe",decimals=3)

In [ ]:
# Plot the relative Sharpe performance for ensemble models against OLS for dataset 3
plot_metric_heatmap_regular(Sharpe_grid3 - Sharpe_grid_OLS3,Temperatures, Transaction_Costs_bps,
    "Sharpe Performance: Ensemble vs. OLS - Dataset 3","Sharpe Difference",decimals=3)

In [ ]:
# Plot the absolute Sharpe performance for ensemble models for dataset 3
plot_metric_heatmap_regular_abs(
    Sharpe_grid3 - Sharpe_grid_FF33,
    Temperatures, Transaction_Costs_bps,"Sharpe Performance: Ensemble vs. FF3 - Dataset 3",
    "Sharpe Difference",decimals=3)

In [ ]:
# Plot the relative Sharpe performance for ensemble models against EN for dataset 3
plot_metric_heatmap_regular(Sharpe_grid3 - Sharpe_grid_EN3,Temperatures, Transaction_Costs_bps,
    "Sharpe Performance: Ensemble vs. EN - Dataset 3","Sharpe Difference",decimals=3)

In [ ]:
# Plot the absolute Turnover performance for ensemble models for dataset 3
plot_metric_heatmap_reversed_abs(Turnover_grid3,Temperatures, Transaction_Costs_bps,
    "Average Portfolio Turnover Ensemble - Dataset 3","Average Turnover",decimals=3)

In [ ]:
# Plot the relative Turnover performance for ensemble models against OLS for dataset 3
plot_metric_heatmap_reversed(Turnover_grid3 - Turnover_grid_OLS3,Temperatures, Transaction_Costs_bps,
    "Average Portfolio Turnover Ensemble vs. OLS - Dataset 3","Average Turnover",decimals=3)

In [ ]:
# Plot the relative Turnover performance for ensemble models against FF3 for dataset 3
plot_metric_heatmap_reversed(Turnover_grid3 - Turnover_grid_FF33,Temperatures, Transaction_Costs_bps,
    "Average Portfolio Turnover: Ensemble vs. FF3 - Dataset 3","Average Turnover",decimals=3)

In [ ]:
# Plot the relative Turnover performance for ensemble models against EN for dataset 3
plot_metric_heatmap_reversed(Turnover_grid3 - Turnover_grid_EN3,Temperatures, Transaction_Costs_bps,
    "Average Portfolio Turnover: Ensemble vs. EN - Dataset 3","Average Turnover",decimals=3)

In [ ]:
# Compute the average Sharpe performance across all three datasets for ensemble and base models

Sharpe_grid_average = (Sharpe_grid1 + Sharpe_grid2 + Sharpe_grid3) / 3
Sharpe_grid_OLS_average = (Sharpe_grid_OLS1 + Sharpe_grid_OLS2 + Sharpe_grid_OLS3) / 3
Sharpe_grid_FF3_average = (Sharpe_grid_FF31 + Sharpe_grid_FF32 + Sharpe_grid_FF33) / 3
Sharpe_grid_EN_average = (Sharpe_grid_EN1 + Sharpe_grid_EN2 + Sharpe_grid_EN3) / 3

# Compute the average Turnover performance across all three datasets for ensemble models

Turnover_grid_average = (Turnover_grid1 + Turnover_grid2 + Turnover_grid3) / 3
Turnover_grid_OLS_average = (Turnover_grid_OLS1 + Turnover_grid_OLS2 + Turnover_grid_OLS3) / 3
Turnover_grid_FF3_average = (Turnover_grid_FF31 + Turnover_grid_FF32 + Turnover_grid_FF33) / 3
Turnover_grid_EN_average = (Turnover_grid_EN1 + Turnover_grid_EN2 + Turnover_grid_EN3) / 3

# Compute the average number of extreme weights across all three datasets for ensemble models and base models
Num_Ext_grid_average = (Num_Ext_grid1 + Num_Ext_grid2 + Num_Ext_grid3) / 3

# Compute the average mean return and standard deviation of return across all three datasets for ensemble models
# and base models
Mean_return_grid_average = (Mean_return_grid1 + Mean_return_grid2 + Mean_return_grid3) / 3
Mean_return_grid_OLS_average = (Mean_return_grid_OLS1 + Mean_return_grid_OLS2 + Mean_return_grid_OLS3) / 3
Mean_return_grid_FF3_average = (Mean_return_grid_FF31 + Mean_return_grid_FF32 + Mean_return_grid_FF33) / 3
Mean_return_grid_EN_average = (Mean_return_grid_EN1 + Mean_return_grid_EN2 + Mean_return_grid_EN3) / 3

# Compute the average standard deviation of return across all three datasets for ensemble models and base models
std_return_grid_average = (std_return_grid1 + std_return_grid2 + std_return_grid3) / 3
std_return_grid_OLS_average = (std_return_grid_OLS1 + std_return_grid_OLS2 + std_return_grid_OLS3) / 3
std_return_grid_FF3_average = (std_return_grid_FF31 + std_return_grid_FF32 + std_return_grid_FF33) / 3
std_return_grid_EN_average = (std_return_grid_EN1 + std_return_grid_EN2 + std_return_grid_EN3) / 3


In [ ]:
# Aggregate across transaction costs to compare the temperature performance of the ensemble model
# against the equal-weighted ensemble model
temp_Sharpe_compare_average = np.mean(Sharpe_grid_average, axis=1)
temp_Turnover_compare_average = np.mean(Turnover_grid_average, axis=1)
temp_Mean_return_compare_average = np.mean(Mean_return_grid_average, axis=1)
temp_std_return_compare_average = np.mean(std_return_grid_average, axis=1)


In [ ]:
plot_metric_heatmap_regular_temp(temp_Sharpe_compare_average,Temperatures,
    "Ensemble Sharpe Comparison by Temperature","Sharpe",decimals=3)

In [ ]:
plot_metric_heatmap_regular_temp_reversed(temp_Turnover_compare_average,Temperatures,
    "Ensemble Turnover Comparison by Temperature","Average Turnover",decimals=4)

In [ ]:
plot_metric_heatmap_regular_abs(Sharpe_grid_average,Temperatures, Transaction_Costs_bps,
    "Sharpe Performance: Ensemble - Aggregated","Sharpe Difference",decimals=3)

In [ ]:
plot_metric_heatmap_regular(Sharpe_grid_average - Sharpe_grid_OLS_average,Temperatures, Transaction_Costs_bps,
    "Sharpe Performance: Ensemble vs. OLS Sharpe - Aggregated","Sharpe Difference",decimals=3)

In [ ]:
plot_metric_heatmap_regular(Sharpe_grid_average - Sharpe_grid_FF3_average,Temperatures, Transaction_Costs_bps,
    "Sharpe Performance: Ensemble vs. FF3 Sharpe - Aggregated","Sharpe Difference",decimals=3)

In [ ]:
plot_metric_heatmap_regular(Sharpe_grid_average - Sharpe_grid_EN_average,Temperatures, Transaction_Costs_bps,
    "Sharpe Performance: Ensemble vs. EN Sharpe - Aggregated","Sharpe Difference",decimals=3)

In [ ]:
plot_metric_heatmap_reversed_abs(Turnover_grid_average,Temperatures, Transaction_Costs_bps,
    "Average Portfolio Turnover: Ensemble - Aggregated","Average Turnover",decimals=3)

In [ ]:
plot_metric_heatmap_reversed(Turnover_grid_average - Turnover_grid_OLS_average,Temperatures, Transaction_Costs_bps,
    "Turnover Performance: Ensemble vs. OLS Turnover - Aggregated","Turnover Difference",decimals=3)

In [ ]:
plot_metric_heatmap_reversed(Turnover_grid_average - Turnover_grid_FF3_average,Temperatures, Transaction_Costs_bps,
    "Turnover Performance: Ensemble vs. FF3 Turnover - Aggregated","Turnover Difference",decimals=3)

In [ ]:
plot_metric_heatmap_reversed(Turnover_grid_average - Turnover_grid_EN_average,Temperatures, Transaction_Costs_bps,
                             "Turnover Performance: Ensemble vs. EN Turnover - Aggregated","Turnover Difference",decimals=3)

In [ ]:
plot_metric_heatmap_reversed(Num_Ext_grid_average,Temperatures, Transaction_Costs_bps,
    "Extreme Weightings - Ensemble Aggregated","Weightings with a Base Model Weight <= 0.1",decimals=3)

In [ ]:
plot_metric_heatmap_regular(Mean_return_grid_average - Mean_return_grid_OLS_average,Temperatures, Transaction_Costs_bps,
    "Mean Return Performance: Ensemble vs. OLS Mean Return - Aggregated","Mean Return Difference",decimals=3)